# KN1GHT Evaluation Harness

Benchmarks KN1GHT (SFT-v5 and DPO) against chess-specialist and frontier LLMs across three phases:

- **Phase A** — 20 opening positions × 5 generations: legality rate, centipawn loss, blunder rate
- **Phase C** — 10 tactical puzzle positions × 5 generations: top-1 accuracy (generalization test)
- **Phase B** — Deep dive for any model that beats KN1GHT DPO on centipawn loss in Phase A

See `PLAN.md` → Step 4 for full rationale and model selection decisions.


## 1. Setup


In [ ]:
import os
import re
import sys
import json
import random
import shutil
import warnings
from pathlib import Path
from typing import Optional

import chess
import chess.engine
import chess.pgn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from openai import OpenAI

# KN1GHT components live in scripts/train.py
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from train import ChessGPT, ModelConfig, TOKENIZER_PATH, CHESS_OPENINGS

warnings.filterwarnings("ignore", category=UserWarning)
print("Imports OK")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
MODELS_DIR = ROOT / ".data" / "models"
KNIGHT_SFT = MODELS_DIR / "kn1ght-sft-v5" / "ckpt_005000.pt"
KNIGHT_DPO = MODELS_DIR / "kn1ght-dpo" / "ckpt_000300.pt"

# Stockfish — tries common install locations then falls back to PATH
STOCKFISH_CANDIDATES = [
    "/opt/homebrew/bin/stockfish",
    "/usr/local/bin/stockfish",
    shutil.which("stockfish") or "",
]
STOCKFISH_PATH = next((p for p in STOCKFISH_CANDIDATES if p and Path(p).exists()), None)
if STOCKFISH_PATH is None:
    raise RuntimeError("Stockfish not found — install via: brew install stockfish")
print(f"Stockfish: {STOCKFISH_PATH}")

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

# ── Evaluation hyperparameters ────────────────────────────────────────────────
N_GENS_A = 5  # generations per position — Phase A
N_POS_A = 20  # probe positions — Phase A
N_GENS_C = 5  # generations per puzzle — Phase C
STOCKFISH_DEPTH = 12
BLUNDER_THRESHOLD = 200  # centipawns
TEMPERATURE = 0.8
TOP_K = 40

## 2. Helper Functions


In [ ]:
# ── PGN utilities ──────────────────────────────────────────────────────────────


def parse_half_moves(pgn_text: str) -> list[str]:
    """Extract flat list of SAN half-moves from a PGN string."""
    clean = pgn_text.replace("[g_start]", "").replace("[g_end]", "").strip()
    clean = re.sub(r"\s*(1-0|0-1|1/2-1/2|\*)\s*$", "", clean).strip()
    no_nums = re.sub(r"\d+\.+", " ", clean)
    return [m.strip() for m in no_nums.split() if m.strip()]


def moves_to_pgn(half_moves: list[str]) -> str:
    """Convert flat SAN list → numbered PGN string."""
    parts = []
    for i, move in enumerate(half_moves):
        if i % 2 == 0:
            parts.append(f"{i // 2 + 1}.{move}")
        else:
            parts.append(move)
    return " ".join(parts)


def pgn_to_board(pgn_text: str) -> chess.Board:
    """Replay PGN moves onto a Board; raise ValueError on illegal move."""
    board = chess.Board()
    for san in parse_half_moves(pgn_text):
        board.push_san(san)
    return board


def extract_next_move(prefix_pgn: str, generated_text: str) -> Optional[str]:
    """
    Given the original prefix PGN and the model's full generated text
    (which may include the prefix), return the first new SAN move.
    """
    prefix_moves = parse_half_moves(prefix_pgn)
    all_moves = parse_half_moves(generated_text)
    if len(all_moves) > len(prefix_moves):
        return all_moves[len(prefix_moves)]
    return None


def is_legal_san(board: chess.Board, san: str) -> bool:
    """Return True if `san` is a legal move in `board`."""
    if not san:
        return False
    try:
        move = board.parse_san(san)
        return move in board.legal_moves
    except Exception:
        return False

In [ ]:
# ── Stockfish evaluation ───────────────────────────────────────────────────────

MATE_CP = 10_000  # centipawn value assigned to forced mate


def _score_relative(info) -> int:
    """Extract centipawn score from Stockfish info (from side-to-move perspective)."""
    score = info["score"].relative
    return score.score(mate_score=MATE_CP)


def get_cpl(board: chess.Board, move_san: str, engine, depth: int = STOCKFISH_DEPTH):
    """
    Return (cpl, is_legal) where:
      cpl        — centipawn loss vs Stockfish best (>= 0, or None if illegal)
      is_legal   — whether the move was legal
    """
    if not move_san:
        return None, False

    # Best-move evaluation (side to move's perspective)
    best_info = engine.analyse(board, chess.engine.Limit(depth=depth))
    best_cp = _score_relative(best_info)

    try:
        move = board.parse_san(move_san)
    except Exception:
        return None, False

    if move not in board.legal_moves:
        return None, False

    # Evaluate position after candidate move (now opponent to move → negate)
    board.push(move)
    cand_info = engine.analyse(board, chess.engine.Limit(depth=depth))
    cand_cp = -_score_relative(cand_info)  # flip to original side
    board.pop()

    cpl = max(0, best_cp - cand_cp)
    return cpl, True

## 3. Load Models


In [ ]:
# ── KN1GHT ────────────────────────────────────────────────────────────────────


def load_kn1ght(path: Path, device: torch.device) -> tuple[ChessGPT, Tokenizer]:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ModelConfig(**ckpt["model_cfg"])
    model = ChessGPT(cfg)
    model.load_state_dict(ckpt["model_state"])
    model.eval().to(device)
    tok = Tokenizer.from_file(str(TOKENIZER_PATH))
    print(f"Loaded KN1GHT from {path.name}  ({model.num_params / 1e6:.1f}M params)")
    return model, tok


@torch.no_grad()
def kn1ght_next_move(
    model: ChessGPT,
    tokenizer: Tokenizer,
    prefix_pgn: str,
    temperature: float = TEMPERATURE,
    top_k: int = TOP_K,
    device: torch.device = DEVICE,
) -> Optional[str]:
    """Generate and return the next SAN move from a KN1GHT model."""
    text = f"[g_start]{prefix_pgn}"
    input_ids = tokenizer.encode(text).ids
    input_len = len(input_ids)

    idx = torch.tensor([input_ids], dtype=torch.long).to(device)
    out = model.generate(idx, max_new_tokens=24, temperature=temperature, top_k=top_k)
    new_ids = out[0, input_len:].tolist()
    new_text = tokenizer.decode(new_ids)
    full_text = f"[g_start]{prefix_pgn} {new_text}"
    return extract_next_move(prefix_pgn, full_text)


kn1ght_sft, _tok_sft = load_kn1ght(KNIGHT_SFT, DEVICE)
kn1ght_dpo, _tok_dpo = load_kn1ght(KNIGHT_DPO, DEVICE)
# Both share the same tokenizer
kn1ght_tokenizer = _tok_sft

In [ ]:
# ── HuggingFace chess LMs ──────────────────────────────────────────────────────
# Downloads on first run (~400 MB each); cached to HF cache afterward.
# chessgpt2-medium-smaller_pgn (mlabonne) is no longer public; using
# mlabonne/chesspythia-70m instead (Pythia-70m fine-tuned on chess PGN, same author).

from transformers import GenerationConfig

HF_CHESS_MODELS = {
    "chessgpt-base-v1": "Waterhorse/chessgpt-base-v1",
    "chesspythia-70m": "mlabonne/chesspythia-70m",
}

hf_pipes: dict = {}

for model_key, hf_id in HF_CHESS_MODELS.items():
    try:
        pipe = pipeline(
            "text-generation",
            model=hf_id,
            device=DEVICE
            if DEVICE.type != "mps"
            else -1,  # HF pipelines use -1 for CPU
            dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        )
        hf_pipes[model_key] = pipe
        print(f"Loaded {model_key}")
    except Exception as e:
        print(f"WARNING: could not load {model_key}: {e}")
        hf_pipes[model_key] = None


def hf_next_move(
    pipe, prefix_pgn: str, temperature: float = TEMPERATURE
) -> Optional[str]:
    """Generate next SAN move from a HuggingFace text-generation pipeline."""
    if pipe is None:
        return None
    try:
        # Pass a GenerationConfig object so it fully overrides any baked-in
        # generation_config.json (e.g. chessgpt-base-v1 ships max_length=20).
        gen_cfg = GenerationConfig(
            max_new_tokens=24,
            do_sample=True,
            temperature=temperature,
            top_k=TOP_K,
            pad_token_id=pipe.tokenizer.eos_token_id,
        )
        result = pipe(prefix_pgn, generation_config=gen_cfg)
        generated = result[0]["generated_text"]
        return extract_next_move(prefix_pgn, generated)
    except Exception:
        return None

In [ ]:
# ── Frontier LLM clients ───────────────────────────────────────────────────────
# OpenRouter: unified API for all cloud models (including gpt-oss-20b).

OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY", "")

openrouter_client = (
    OpenAI(api_key=OPENROUTER_KEY, base_url="https://openrouter.ai/api/v1")
    if OPENROUTER_KEY
    else None
)

# Model IDs on OpenRouter.
# max_tokens: thinking models need a large budget to cover reasoning tokens before
# emitting output. Non-thinking models only need enough for a single SAN move.
FRONTIER_MODELS = {
    # completion-style (raw PGN, no system prompt)
    "gpt-3.5-turbo-instruct": {
        "client": openrouter_client,
        "id": "openai/gpt-3.5-turbo-instruct",
        "mode": "completion",
        "max_tokens": 6,
    },
    "gpt-oss-20b": {
        "client": openrouter_client,
        "id": "openai/gpt-oss-20b",
        "mode": "completion",
        "max_tokens": 6,
    },
    # chat-style
    "gpt-4.1-nano": {
        "client": openrouter_client,
        "id": "openai/gpt-4.1-nano",
        "mode": "chat",
        "max_tokens": 16,  # Azure routing enforces a minimum of 16
    },
    "gpt-5-nano": {
        "client": openrouter_client,
        "id": "openai/gpt-5-nano",
        "mode": "chat",
        "max_tokens": 512,  # thinking model: reasoning tokens count against max_tokens
    },
    "claude-haiku-4.5": {
        "client": openrouter_client,
        "id": "anthropic/claude-haiku-4-5",
        "mode": "chat",
        "max_tokens": 16,
    },
    "gemini-3.1-flash-lite-preview": {
        "client": openrouter_client,
        "id": "google/gemini-3.1-flash-lite-preview",
        "mode": "chat",
        "max_tokens": 16,
    },
    "deepseek-v3": {
        "client": openrouter_client,
        "id": "deepseek/deepseek-chat-v3-0324",
        "mode": "chat",
        "max_tokens": 16,
    },
}

_CHESS_SYSTEM = "You play chess. Reply with only the next move in SAN notation."


def frontier_next_move(
    model_key: str, prefix_pgn: str, temperature: float = TEMPERATURE
) -> Optional[str]:
    """Generate next SAN move from a frontier LLM."""
    spec = FRONTIER_MODELS.get(model_key)
    if spec is None:
        return None
    client = spec["client"]
    if client is None:
        return None
    try:
        if spec["mode"] == "completion":
            resp = client.completions.create(
                model=spec["id"],
                prompt=prefix_pgn + " ",
                max_tokens=spec["max_tokens"],
                temperature=temperature,
            )
            raw = resp.choices[0].text.strip()
        else:
            resp = client.chat.completions.create(
                model=spec["id"],
                messages=[
                    {"role": "system", "content": _CHESS_SYSTEM},
                    {"role": "user", "content": prefix_pgn},
                ],
                max_tokens=spec["max_tokens"],
                temperature=temperature,
            )
            raw = (resp.choices[0].message.content or "").strip()
        # Extract first SAN-shaped token from the raw response
        tokens = raw.split()
        for tok in tokens:
            tok = re.sub(r"^\d+\.+", "", tok).strip()
            if tok:
                return tok
        return None
    except Exception as e:
        print(f"  [{model_key}] API error: {e}")
        return None


print("Frontier clients configured")
if not OPENROUTER_KEY:
    print(
        "  WARNING: OPENROUTER_API_KEY not set — cloud frontier models will be skipped"
    )

In [ ]:
# ── Stockfish engine ───────────────────────────────────────────────────────────
engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
engine.configure({"Threads": 2, "Hash": 64})
print(f"Stockfish engine ready")

In [ ]:
# ── Unified model interface ────────────────────────────────────────────────────
# Each entry: name → callable(prefix_pgn: str) -> Optional[str]

ALL_MODELS = {
    # KN1GHT
    "kn1ght-sft-v5": lambda pgn: kn1ght_next_move(kn1ght_sft, kn1ght_tokenizer, pgn),
    "kn1ght-dpo": lambda pgn: kn1ght_next_move(kn1ght_dpo, kn1ght_tokenizer, pgn),
    # HF chess models
    **{k: (lambda pgn, pipe=p: hf_next_move(pipe, pgn)) for k, p in hf_pipes.items()},
    # Frontier LLMs
    **{k: (lambda pgn, key=k: frontier_next_move(key, pgn)) for k in FRONTIER_MODELS},
}

# Chess-specialist models only (no frontier LLMs) — used for legality-rate metric
SPECIALIST_MODELS = ["kn1ght-sft-v5", "kn1ght-dpo"] + list(hf_pipes.keys())

print(f"Models registered: {list(ALL_MODELS.keys())}")

## 4. Probe Positions (Phase A)


In [ ]:
def generate_probe_positions(
    openings: list[tuple[str, str]],
    n_total: int = N_POS_A,
    plies: tuple[int, ...] = (4, 8),
    seed: int = 42,
) -> list[dict]:
    """
    Sample `n_total // len(plies)` openings that have enough moves, then
    generate one probe position per ply depth for each sampled opening.
    """
    rng = random.Random(seed)
    n_per_ply = n_total // len(plies)
    candidates = [
        (name, pgn)
        for name, pgn in openings
        if len(parse_half_moves(pgn)) >= max(plies)
    ]
    sampled = rng.sample(candidates, min(n_per_ply, len(candidates)))

    positions = []
    for name, pgn in sampled:
        half_moves = parse_half_moves(pgn)
        for ply in plies:
            probe_pgn = moves_to_pgn(half_moves[:ply])
            positions.append(
                {
                    "name": f"{name} ({ply}ply)",
                    "pgn": probe_pgn,
                    "ply": ply,
                    "family": name,
                }
            )
    return positions[:n_total]


probe_positions = generate_probe_positions(CHESS_OPENINGS)
print(f"Phase A: {len(probe_positions)} probe positions")
for p in probe_positions[:4]:
    print(f"  {p['name']:50s}  {p['pgn'][:60]}")

## 5. Puzzle Positions (Phase C — Generalization)

Puzzles are drawn from [wtharvey.com](https://wtharvey.com/m8n2.txt) — mate-in-2
positions from real tournament games spanning 1840–2020.
They are pre-fetched and cached to `.data/puzzles/wtharvey_fen_puzzles.json`
by `notebooks/build-puzzle-dataset.ipynb`.

**Why FEN puzzles are a useful generalization test:**
FEN notation was never in KN1GHT's training data — its tokenizer decomposes
FEN character-by-character with no chess meaning. KN1GHT's expected score is
**~0%**, which is honest and informative: the gap vs frontier LLMs is the
clearest evidence of what PGN-only training _doesn't_ buy you.

**Phase C' (PGN puzzles) is in the build-puzzle-dataset notebook.** Those puzzles
present tactical positions in KN1GHT's native PGN format — full game history up to
a middlegame tactical shot. Load them with:

```python
pgn_puzzle_file = ROOT / ".data" / "puzzles" / "pgn_puzzles.jsonl"
```


In [ ]:
# ── Load pre-cached FEN puzzles (Phase C) ────────────────────────────────────
# Run notebooks/build-puzzle-dataset.ipynb first to populate these files.
import json as _json

FEN_PUZZLE_FILE = ROOT / ".data" / "puzzles" / "wtharvey_fen_puzzles.json"

if FEN_PUZZLE_FILE.exists():
    FEN_PUZZLES = _json.loads(FEN_PUZZLE_FILE.read_text())
    print(f"Loaded {len(FEN_PUZZLES)} FEN puzzles from cache")
else:
    raise FileNotFoundError(
        f"{FEN_PUZZLE_FILE} not found. "
        "Run notebooks/build-puzzle-dataset.ipynb to build the puzzle cache."
    )

print()
for pz in FEN_PUZZLES:
    print(f"  {pz['citation']}")
    print(f"    FEN:      {pz['fen']}")
    print(f"    Best:     {pz['best_move']}")
    print()

### 5b. PGN Puzzle Positions (Phase C')

Tactical positions from the Lichess puzzle database, presented as full PGN game
context — KN1GHT's native format. Built by `notebooks/build-puzzle-dataset.ipynb`.

Unlike Phase C (FEN), KN1GHT can actually process these. Phase C' answers:
_when the model reaches a tactically sharp position in PGN context, does it find
the correct move?_


In [ ]:
# ── Load pre-built PGN puzzle dataset (Phase C') ─────────────────────────────
PGN_PUZZLE_FILE = ROOT / ".data" / "puzzles" / "pgn_puzzles.jsonl"

if not PGN_PUZZLE_FILE.exists():
    raise FileNotFoundError(
        f"{PGN_PUZZLE_FILE} not found. "
        "Run notebooks/build-puzzle-dataset.ipynb to build the puzzle dataset."
    )

with open(PGN_PUZZLE_FILE) as _f:
    _all_pgn_puzzles = [json.loads(l) for l in _f if l.strip()]

# Reproducible sample for this eval run — change seed or n to vary
PGN_PUZZLES = random.Random(42).sample(_all_pgn_puzzles, min(20, len(_all_pgn_puzzles)))
print(
    f"Loaded {len(_all_pgn_puzzles)} PGN puzzles total; sampled {len(PGN_PUZZLES)} for eval"
)
print()
for pz in PGN_PUZZLES[:3]:
    print(f"  {pz['puzzle_id']}  rating={pz['rating']}  themes={pz['themes']}")
    print(f"    context: ...{pz['pgn_context'][-60:]}")
    print(f"    answer:  {pz['best_move_san']}")
    print()

## 6. Phase A — Screening (20 positions × 5 gens)


In [ ]:
def run_phase_a(
    positions: list[dict],
    models: dict,
    specialist_models: list[str],
    engine,
    n_gens: int = N_GENS_A,
) -> pd.DataFrame:
    """
    Evaluate each model on each probe position.

    Returns a DataFrame with one row per (model, position).
    Columns: model, position, family, ply, legality_rate, mean_cpl, blunder_rate, n_legal.

    Notes:
    - legality_rate is only meaningful for specialist (PGN continuation) models.
    - CPL is computed only for legal moves.
    """
    rows = []
    total = len(positions) * len(models)
    done = 0

    for pos in positions:
        board = pgn_to_board(pos["pgn"])

        for model_name, gen_fn in models.items():
            done += 1
            print(f"  [{done}/{total}] {model_name:35s} — {pos['name'][:45]}")

            legal_count = 0
            cpls = []
            blunders = 0

            for _ in range(n_gens):
                try:
                    move = gen_fn(pos["pgn"])
                except Exception:
                    move = None

                if is_legal_san(board, move):
                    legal_count += 1
                    cpl, ok = get_cpl(board, move, engine)
                    if ok and cpl is not None:
                        cpls.append(cpl)
                        if cpl >= BLUNDER_THRESHOLD:
                            blunders += 1

            rows.append(
                {
                    "model": model_name,
                    "position": pos["name"],
                    "family": pos["family"],
                    "ply": pos["ply"],
                    "is_specialist": model_name in specialist_models,
                    "n_gens": n_gens,
                    "n_legal": legal_count,
                    "legality_rate": legal_count / n_gens,
                    "mean_cpl": float(np.mean(cpls)) if cpls else None,
                    "blunder_rate": blunders / n_gens,
                }
            )

    return pd.DataFrame(rows)


print("Phase A ready — run next cell to execute")

In [ ]:
# ── Run Phase A ────────────────────────────────────────────────────────────────
# This cell takes a while — each model × position × generation × Stockfish call.
# Expect ~5-10 minutes depending on hardware and API latency.

phase_a_df = run_phase_a(
    probe_positions,
    ALL_MODELS,
    SPECIALIST_MODELS,
    engine,
)

# Save results so we don't have to rerun
phase_a_df.to_csv(ROOT / ".data" / "eval_phase_a.csv", index=False)
print("Phase A complete — results saved to .data/eval_phase_a.csv")

In [ ]:
# ── Phase A results — aggregated by model ─────────────────────────────────────

phase_a_summary = (
    phase_a_df.groupby("model")
    .agg(
        legality_rate=("legality_rate", "mean"),
        mean_cpl=("mean_cpl", "mean"),
        blunder_rate=("blunder_rate", "mean"),
        is_specialist=("is_specialist", "first"),
    )
    .reset_index()
    .sort_values("mean_cpl")
)


def fmt_pct(x):
    return f"{x * 100:.1f}%" if x is not None and not np.isnan(x) else "—"


def fmt_cp(x):
    return f"{x:.0f}" if x is not None and not np.isnan(x) else "—"


display_df = phase_a_summary[
    ["model", "legality_rate", "mean_cpl", "blunder_rate", "is_specialist"]
].copy()
display_df["legality_rate"] = display_df["legality_rate"].apply(fmt_pct)
display_df["mean_cpl"] = display_df["mean_cpl"].apply(fmt_cp)
display_df["blunder_rate"] = display_df["blunder_rate"].apply(fmt_pct)
display_df.columns = [
    "Model",
    "Legality Rate",
    "Mean CPL ↓",
    "Blunder Rate",
    "Specialist?",
]
print("Phase A — aggregated results (sorted by CPL):")
display_df

In [ ]:
# ── Phase A plots ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Phase A Screening Results", fontsize=14, fontweight="bold")

models_ordered = phase_a_summary["model"].tolist()

# Centipawn Loss
ax = axes[0]
cpls = phase_a_summary["mean_cpl"].fillna(0)
bars = ax.barh(
    models_ordered,
    cpls,
    color=["#4CAF50" if m.startswith("kn1ght") else "#2196F3" for m in models_ordered],
)
ax.set_xlabel("Mean CPL (lower = better)")
ax.set_title("Centipawn Loss")
ax.invert_yaxis()

# Legality Rate (specialist models only)
ax = axes[1]
spec_df = phase_a_summary[phase_a_summary["is_specialist"]]
ax.barh(spec_df["model"], spec_df["legality_rate"] * 100, color="#9C27B0")
ax.set_xlabel("Legality Rate (%) — specialist models only")
ax.set_title("Legal Move Rate")
ax.set_xlim(0, 100)
ax.invert_yaxis()

# Blunder Rate
ax = axes[2]
ax.barh(models_ordered, phase_a_summary["blunder_rate"] * 100, color="#F44336")
ax.set_xlabel("Blunder Rate (%) — CPL > 200")
ax.set_title("Blunder Rate")
ax.set_xlim(0, 100)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(ROOT / ".data" / "eval_phase_a.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Phase C — Puzzle Task (Generalization)

Tests whether the model has internalized chess strategy beyond the opening patterns in its training data.
Metric: **top-1 accuracy** — did the model play the theoretically best move?

Any accuracy meaningfully above the random baseline (~1/30 legal moves ≈ 3%) is genuine generalization.


In [ ]:
# ── FEN puzzle solve functions ─────────────────────────────────────────────────
# Each function receives a FEN string and returns a move string (or None).

_FEN_SYSTEM = (
    "You are a chess engine. Given a position in FEN notation, "
    "respond with ONLY the best move in SAN notation (e.g. 'Nf6+' or 'Rxd8#'). "
    "No explanation, no punctuation, just the move."
)


def _extract_move_token(text: str) -> Optional[str]:
    """Pull the first plausible SAN token out of a raw string."""
    # Strip leading move numbers (e.g. "1. " or "1...")
    text = re.sub(r"^\d+\.+\s*", "", text.strip())
    tokens = text.split()
    return tokens[0].strip() if tokens else None


@torch.no_grad()
def kn1ght_solve_fen(
    model: ChessGPT,
    tokenizer: Tokenizer,
    fen: str,
    device: torch.device = DEVICE,
    temperature: float = TEMPERATURE,
    top_k: int = TOP_K,
) -> Optional[str]:
    """
    Attempt to solve a FEN puzzle with KN1GHT.

    The model was trained on PGN text and has never seen FEN notation.
    FEN will be tokenised as arbitrary characters, so the output is
    expected to be random / nonsensical.  We extract whatever the first
    generated token looks like and return it for honest scoring.
    """
    text = f"[g_start]{fen} "
    input_ids = tokenizer.encode(text).ids
    input_len = len(input_ids)
    idx = torch.tensor([input_ids], dtype=torch.long).to(device)
    out = model.generate(idx, max_new_tokens=10, temperature=temperature, top_k=top_k)
    new_text = tokenizer.decode(out[0, input_len:].tolist()).strip()
    return _extract_move_token(new_text)


def hf_solve_fen(pipe, fen: str, temperature: float = TEMPERATURE) -> Optional[str]:
    """Attempt to solve a FEN puzzle with a HuggingFace text-generation pipeline."""
    if pipe is None:
        return None
    try:
        gen_cfg = GenerationConfig(
            max_new_tokens=10,
            do_sample=True,
            temperature=temperature,
            pad_token_id=pipe.tokenizer.eos_token_id,
        )
        result = pipe(fen, generation_config=gen_cfg)
        suffix = result[0]["generated_text"][len(fen) :].strip()
        return _extract_move_token(suffix)
    except Exception:
        return None


def frontier_solve_fen(
    model_key: str, fen: str, temperature: float = TEMPERATURE
) -> Optional[str]:
    """Solve a FEN puzzle using a frontier LLM."""
    spec = FRONTIER_MODELS.get(model_key)
    if spec is None or spec["client"] is None:
        return None
    try:
        if spec["mode"] == "completion":
            resp = spec["client"].completions.create(
                model=spec["id"],
                prompt=f"Position (FEN): {fen}\nBest move: ",
                max_tokens=spec["max_tokens"],
                temperature=temperature,
            )
            raw = resp.choices[0].text
        else:
            resp = spec["client"].chat.completions.create(
                model=spec["id"],
                messages=[
                    {"role": "system", "content": _FEN_SYSTEM},
                    {"role": "user", "content": fen},
                ],
                max_tokens=spec["max_tokens"],
                temperature=temperature,
            )
            raw = resp.choices[0].message.content or ""
        return _extract_move_token(raw.strip())
    except Exception as e:
        print(f"  [{model_key}] FEN API error: {e}")
        return None


# ── Unified FEN model registry ─────────────────────────────────────────────────
FEN_MODELS = {
    "kn1ght-sft-v5": lambda fen: kn1ght_solve_fen(kn1ght_sft, kn1ght_tokenizer, fen),
    "kn1ght-dpo": lambda fen: kn1ght_solve_fen(kn1ght_dpo, kn1ght_tokenizer, fen),
    **{k: (lambda fen, pipe=p: hf_solve_fen(pipe, fen)) for k, p in hf_pipes.items()},
    **{k: (lambda fen, key=k: frontier_solve_fen(key, fen)) for k in FRONTIER_MODELS},
}

print(f"FEN model registry: {list(FEN_MODELS.keys())}")

In [ ]:
def run_fen_puzzle_eval(
    puzzles: list[dict],
    models: dict,
    n_gens: int = N_GENS_C,
) -> pd.DataFrame:
    """
    Evaluate each model on each FEN puzzle.

    A generation is a "hit" if the move it produces matches the expected
    first move of the mate-in-2 solution (check/mate annotations stripped
    for comparison, since models vary in annotation style).

    Legality is also checked — a move that isn't legal in the position
    can't be a hit.
    """

    def normalise(san: str) -> str:
        return san.rstrip("+#").strip() if san else ""

    rows = []
    total = len(puzzles) * len(models)
    done = 0

    for pz in puzzles:
        board = chess.Board(pz["fen"])
        expected = normalise(pz["best_move"])
        accepted = {pz["best_move"], expected}  # with and without annotation

        for model_name, solve_fn in models.items():
            done += 1
            print(f"  [{done}/{total}] {model_name:35s} — {pz['citation'][:50]}")

            hits = 0
            legal = 0
            moves = []

            for _ in range(n_gens):
                try:
                    move = solve_fn(pz["fen"])
                except Exception:
                    move = None

                is_legal = is_legal_san(board, move) if move else False
                is_hit = bool(
                    move
                    and is_legal
                    and (move in accepted or normalise(move) == expected)
                )
                if is_legal:
                    legal += 1
                if is_hit:
                    hits += 1
                moves.append(move)

            rows.append(
                {
                    "model": model_name,
                    "citation": pz["citation"],
                    "fen": pz["fen"],
                    "expected": pz["best_move"],
                    "solution": pz["solution"],
                    "n_gens": n_gens,
                    "n_legal": legal,
                    "n_hits": hits,
                    "legality_rate": legal / n_gens,
                    "top1_acc": hits / n_gens,
                    "sample_moves": str(moves),
                }
            )

    return pd.DataFrame(rows)


print("Phase C (FEN) ready — run next cell to execute")

In [ ]:
puzzle_df = run_fen_puzzle_eval(FEN_PUZZLES, FEN_MODELS)

puzzle_df.to_csv(ROOT / ".data" / "eval_phase_c.csv", index=False)
print("Phase C complete — results saved to .data/eval_phase_c.csv")

In [ ]:
# ── Phase C results — aggregated by model ─────────────────────────────────────

puzzle_summary = (
    puzzle_df.groupby("model")
    .agg(top1_acc=("top1_acc", "mean"), legality_rate=("legality_rate", "mean"))
    .reset_index()
    .sort_values("top1_acc", ascending=False)
)

print("Phase C — FEN Puzzle Accuracy (mate-in-2, sorted by top-1 accuracy):")
print()
print(f"  {'Model':<35} {'Top-1 Acc':>10} {'Legality':>10}  Notes")
print("  " + "-" * 70)
for _, row in puzzle_summary.iterrows():
    note = (
        "(PGN model — FEN unsupported)"
        if row["model"].startswith("kn1ght") or "chessgpt" in row["model"]
        else ""
    )
    print(
        f"  {row['model']:<35} {fmt_pct(row['top1_acc']):>10} {fmt_pct(row['legality_rate']):>10}  {note}"
    )

In [ ]:
# ── Phase C plot ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Phase C — FEN Puzzle Accuracy (mate-in-2 from wtharvey.com)",
    fontsize=13,
    fontweight="bold",
)

models_pz = puzzle_summary["model"].tolist()
accs = (puzzle_summary["top1_acc"] * 100).tolist()
legal_rates = (puzzle_summary["legality_rate"] * 100).tolist()


def model_color(name):
    if name.startswith("kn1ght"):
        return "#4CAF50"
    if "chessgpt" in name:
        return "#9C27B0"
    return "#2196F3"


colors = [model_color(m) for m in models_pz]

# Top-1 accuracy
ax = axes[0]
ax.barh(models_pz, accs, color=colors)
ax.axvline(
    100 / 30, color="red", linestyle="--", linewidth=1.2, label="~3% random baseline"
)
ax.set_xlabel("Top-1 Accuracy (%)")
ax.set_title("Mate-in-2 Accuracy\n(higher = better generalisation)")
ax.legend(fontsize=8)
ax.invert_yaxis()

# Legality rate — tells us whether the model is even generating chess moves
ax = axes[1]
ax.barh(models_pz, legal_rates, color=colors)
ax.set_xlabel(
    "Legal Move Rate (%)\n(KN1GHT/HF expected ~0% — FEN not in training vocab)"
)
ax.set_title("Legal Move Rate on FEN Input")
ax.set_xlim(0, 100)
ax.invert_yaxis()

from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#4CAF50", label="KN1GHT (PGN model)"),
    Patch(facecolor="#9C27B0", label="HF chess models (PGN)"),
    Patch(facecolor="#2196F3", label="Frontier LLMs"),
]
axes[1].legend(handles=legend_elements, fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(ROOT / ".data" / "eval_phase_c.png", dpi=150, bbox_inches="tight")
plt.show()

## 7b. Phase C' — PGN Puzzle Task

Same tactical positions as Phase C', but presented as PGN game context — the
native format for KN1GHT and the HF chess models.

**Metric:** top-1 accuracy — did the model generate the engine-validated best move  
**Legality** is verified against the puzzle FEN (catches move-number confusion or drift)  
**Expected:** KN1GHT should score meaningfully above the ~3% random baseline here,
unlike Phase C (FEN) where ~0% is expected.


In [ ]:
PHASE_CP_CHECKPOINT = ROOT / ".data" / "eval_phase_cp.csv"
PHASE_CP_COLUMNS = [
    "model",
    "puzzle_id",
    "rating",
    "themes",
    "fen",
    "expected",
    "n_gens",
    "n_legal",
    "n_hits",
    "legality_rate",
    "top1_acc",
    "sample_moves",
]


def run_pgn_puzzle_eval(
    puzzles: list[dict],
    models: dict,
    n_gens: int = N_GENS_C,
    checkpoint_path: Optional[Path] = PHASE_CP_CHECKPOINT,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Evaluate each model on PGN-context puzzles.

    Feeds pgn_context to each model and checks whether it generates the
    engine-validated best move. Legality is verified against the puzzle FEN.
    If a checkpoint CSV exists, completed `(puzzle_id, model)` rows are reused
    so interrupted runs can resume instead of starting over.
    """

    def normalise(san: str) -> str:
        return san.rstrip("+#").strip() if san else ""

    rows = []
    completed_keys = set()
    total = len(puzzles) * len(models)

    checkpoint_label = None
    if checkpoint_path is not None:
        checkpoint_path = Path(checkpoint_path)
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        checkpoint_label = (
            checkpoint_path.relative_to(ROOT)
            if checkpoint_path.is_relative_to(ROOT)
            else checkpoint_path
        )
        if resume and checkpoint_path.exists():
            existing = pd.read_csv(checkpoint_path)
            missing_cols = [c for c in PHASE_CP_COLUMNS if c not in existing.columns]
            if missing_cols:
                print(
                    "Phase C' checkpoint is missing expected columns "
                    f"{missing_cols}; starting fresh."
                )
            else:
                valid_models = set(models)
                valid_puzzles = {pz["puzzle_id"] for pz in puzzles}
                existing = existing[PHASE_CP_COLUMNS].drop_duplicates(
                    subset=["puzzle_id", "model"], keep="last"
                )
                existing = existing[
                    existing["model"].isin(valid_models)
                    & existing["puzzle_id"].isin(valid_puzzles)
                ]
                rows = existing.to_dict("records")
                completed_keys = set(zip(existing["puzzle_id"], existing["model"]))
                print(
                    "Phase C' resume: loaded "
                    f"{len(completed_keys)}/{total} completed rows from "
                    f"{checkpoint_label}"
                )

    done = len(completed_keys)

    for pz in puzzles:
        board = chess.Board(pz["fen"])
        exp_norm = normalise(pz["best_move_san"])
        accepted = {pz["best_move_san"], exp_norm}

        for model_name, solve_fn in models.items():
            key = (pz["puzzle_id"], model_name)
            if key in completed_keys:
                continue

            done += 1
            print(
                f"  [{done}/{total}] {model_name:35s} — {pz['puzzle_id']} "
                f"(rating {pz['rating']}, answer {pz['best_move_san']})"
            )

            hits = 0
            legal = 0
            moves = []

            for _ in range(n_gens):
                try:
                    move = solve_fn(pz["pgn_context"])
                except Exception:
                    move = None

                is_legal = is_legal_san(board, move) if move else False
                is_hit = bool(
                    move
                    and is_legal
                    and (move in accepted or normalise(move) == exp_norm)
                )
                if is_legal:
                    legal += 1
                if is_hit:
                    hits += 1
                moves.append(move)

            rows.append(
                {
                    "model": model_name,
                    "puzzle_id": pz["puzzle_id"],
                    "rating": pz["rating"],
                    "themes": " ".join(pz["themes"]),
                    "fen": pz["fen"],
                    "expected": pz["best_move_san"],
                    "n_gens": n_gens,
                    "n_legal": legal,
                    "n_hits": hits,
                    "legality_rate": legal / n_gens,
                    "top1_acc": hits / n_gens,
                    "sample_moves": str(moves),
                }
            )
            completed_keys.add(key)

            if checkpoint_path is not None:
                pd.DataFrame(rows, columns=PHASE_CP_COLUMNS).to_csv(
                    checkpoint_path, index=False
                )

    result = pd.DataFrame(rows, columns=PHASE_CP_COLUMNS)
    puzzle_order = {pz["puzzle_id"]: i for i, pz in enumerate(puzzles)}
    model_order = {name: i for i, name in enumerate(models)}
    result = (
        result.assign(
            _puzzle_order=result["puzzle_id"].map(puzzle_order),
            _model_order=result["model"].map(model_order),
        )
        .sort_values(["_puzzle_order", "_model_order"])
        .drop(columns=["_puzzle_order", "_model_order"])
        .reset_index(drop=True)
    )

    if checkpoint_path is not None:
        result.to_csv(checkpoint_path, index=False)

    return result


print("Phase C' (PGN) ready — rerun the next cell to resume from checkpoint if needed")


In [ ]:
# Rerun this cell after an interruption to resume from .data/eval_phase_cp.csv.
# Delete that file or pass resume=False in the call below to force a full rerun.

pgn_puzzle_df = run_pgn_puzzle_eval(
    PGN_PUZZLES,
    ALL_MODELS,
    checkpoint_path=PHASE_CP_CHECKPOINT,
    resume=True,
)

pgn_puzzle_df.to_csv(PHASE_CP_CHECKPOINT, index=False)
checkpoint_label = (
    PHASE_CP_CHECKPOINT.relative_to(ROOT)
    if PHASE_CP_CHECKPOINT.is_relative_to(ROOT)
    else PHASE_CP_CHECKPOINT
)
print(f"Phase C' complete — results saved to {checkpoint_label}")


In [ ]:
# ── Phase C' results — aggregated by model ────────────────────────────────────

pgn_puzzle_summary = (
    pgn_puzzle_df.groupby("model")
    .agg(top1_acc=("top1_acc", "mean"), legality_rate=("legality_rate", "mean"))
    .reset_index()
    .sort_values("top1_acc", ascending=False)
)

print("Phase C' — PGN Puzzle Accuracy (Lichess middlegame, sorted by top-1 accuracy):")
print()
print(f"  {'Model':<35} {'Top-1 Acc':>10} {'Legality':>10}")
print("  " + "-" * 60)
for _, row in pgn_puzzle_summary.iterrows():
    print(
        f"  {row['model']:<35} {fmt_pct(row['top1_acc']):>10} {fmt_pct(row['legality_rate']):>10}"
    )

In [ ]:
# ── Phase C' plot ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Phase C' — PGN Puzzle Accuracy (Lichess middlegame tactics)",
    fontsize=13,
    fontweight="bold",
)

models_pz = pgn_puzzle_summary["model"].tolist()
accs = (pgn_puzzle_summary["top1_acc"] * 100).tolist()
legal_rates = (pgn_puzzle_summary["legality_rate"] * 100).tolist()
colors = [model_color(m) for m in models_pz]

ax = axes[0]
ax.barh(models_pz, accs, color=colors)
ax.axvline(
    100 / 30, color="red", linestyle="--", linewidth=1.2, label="~3% random baseline"
)
ax.set_xlabel("Top-1 Accuracy (%)")
ax.set_title("Move Accuracy")
ax.legend(fontsize=8)

ax = axes[1]
ax.barh(models_pz, legal_rates, color=colors)
ax.set_xlabel("Legality Rate (%)")
ax.set_title("Legality Rate")
ax.set_xlim(0, 105)

plt.tight_layout()
plt.savefig(ROOT / ".data" / "eval_phase_cp.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Phase B Decision

Promote a model to Phase B if it:

- Beats KN1GHT DPO on mean CPL **or**
- Shows unexpectedly high / low legality relative to its type

Edit the cell below with your Phase A findings before running Phase B.


In [ ]:
# ── Set this based on Phase A results ─────────────────────────────────────────
PHASE_B_MODELS = [
    "gemini-3.1-flash-lite-preview",  # best overall (3.78 CPL) — surprising leader
    "chessgpt-base-v1",  # best chess specialist (4.39 CPL) — specialist ceiling
    "deepseek-v3",  # best non-chess frontier model (5.43 CPL)
    "gpt-3.5-turbo-instruct",  # completion model nearly tied with kn1ght-dpo (7.06 CPL)
]

# Auto-suggest: any model with lower CPL than kn1ght-dpo
kn1ght_dpo_cpl = phase_a_summary.loc[
    phase_a_summary["model"] == "kn1ght-dpo", "mean_cpl"
].values
if len(kn1ght_dpo_cpl) > 0 and not np.isnan(kn1ght_dpo_cpl[0]):
    threshold = kn1ght_dpo_cpl[0]
    candidates = phase_a_summary[
        (phase_a_summary["mean_cpl"] < threshold)
        & (phase_a_summary["model"] != "kn1ght-dpo")
    ]["model"].tolist()
    print(f"KN1GHT DPO mean CPL: {threshold:.0f}")
    print(f"Auto-suggested Phase B candidates: {candidates}")
else:
    print("Run Phase A first.")

print(f"Phase B models set: {PHASE_B_MODELS}")

## 9. Phase B — Deep Dive


In [ ]:
# Only runs if PHASE_B_MODELS is non-empty.

N_GENS_B = 10
N_POS_B = 50

if not PHASE_B_MODELS:
    print("No models promoted to Phase B — skipping.")
else:
    # Restart Stockfish if the engine subprocess died between phases
    try:
        engine.ping()
    except Exception:
        print("Stockfish engine was dead — restarting...")
        try:
            engine.quit()
        except Exception:
            pass
        engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
        engine.configure({"Threads": 2, "Hash": 64})
        print("Stockfish engine restarted")

    # Generate full probe set (all openings with enough moves, 2 plies each)
    full_positions = generate_probe_positions(CHESS_OPENINGS, n_total=N_POS_B, seed=42)
    phase_b_models = {
        k: v for k, v in ALL_MODELS.items() if k in PHASE_B_MODELS + ["kn1ght-dpo"]
    }

    print(
        f"Phase B: {len(full_positions)} positions × {N_GENS_B} gens × {len(phase_b_models)} models"
    )
    phase_b_df = run_phase_a(
        full_positions,
        phase_b_models,
        SPECIALIST_MODELS,
        engine,
        n_gens=N_GENS_B,
    )
    phase_b_df.to_csv(ROOT / ".data" / "eval_phase_b.csv", index=False)

    phase_b_summary = (
        phase_b_df.groupby("model")
        .agg(
            legality_rate=("legality_rate", "mean"),
            mean_cpl=("mean_cpl", "mean"),
            blunder_rate=("blunder_rate", "mean"),
        )
        .reset_index()
        .sort_values("mean_cpl")
    )
    print("Phase B results:")
    display(phase_b_summary)

## 10. Final Summary


In [ ]:
# ── Combined summary table ───────────────────────────────────────────────────

summary = (
    phase_a_summary[["model", "legality_rate", "mean_cpl", "blunder_rate"]]
    .merge(
        puzzle_summary[["model", "top1_acc"]].rename(columns={"top1_acc": "fen_acc"}),
        on="model",
        how="left",
    )
    .merge(
        pgn_puzzle_summary[["model", "top1_acc"]].rename(
            columns={"top1_acc": "pgn_acc"}
        ),
        on="model",
        how="left",
    )
    .sort_values("mean_cpl")
)

print("=" * 92)
print("KN1GHT EVALUATION — FINAL SUMMARY")
print("=" * 92)
print(
    f"{'Model':<35} {'Legality':>9} {'Mean CPL':>9} {'Blunder%':>9} "
    f"{'FEN Pzl%':>9} {'PGN Pzl%':>9}"
)
print("-" * 92)
for _, row in summary.iterrows():
    leg = fmt_pct(row["legality_rate"])
    cpl = fmt_cp(row["mean_cpl"])
    bln = fmt_pct(row["blunder_rate"])
    fen = fmt_pct(row["fen_acc"]) if pd.notna(row.get("fen_acc")) else "—"
    pgn = fmt_pct(row["pgn_acc"]) if pd.notna(row.get("pgn_acc")) else "—"
    print(f"  {row['model']:<33} {leg:>9} {cpl:>9} {bln:>9} {fen:>9} {pgn:>9}")
print("=" * 92)

In [ ]:
# ── Qualitative examples — same 5 positions, all models side by side ───────────

QUALITATIVE_POSITIONS = probe_positions[:5]

print("Qualitative examples — each model's first move in 5 positions:")
print()
for pos in QUALITATIVE_POSITIONS:
    board = pgn_to_board(pos["pgn"])
    best_info = engine.analyse(board, chess.engine.Limit(depth=STOCKFISH_DEPTH))
    sf_best = board.san(best_info["pv"][0]) if "pv" in best_info else "?"

    print(f"Position: {pos['name']}")
    print(f"  PGN:        {pos['pgn']}")
    print(f"  Stockfish:  {sf_best}")
    for model_name, gen_fn in ALL_MODELS.items():
        try:
            move = gen_fn(pos["pgn"])
        except Exception:
            move = "ERROR"
        legal_str = "✓" if is_legal_san(board, move) else "✗"
        print(f"  {model_name:<35} {legal_str}  {move}")
    print()

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
engine.quit()
print("Stockfish engine closed.")